In [7]:
%%writefile practice82.cu

// Реализация обработки массива на GPU с использованием CUDA
// 1. Скопируйте массив данных на GPU.
// 2. Реализуйте ядро CUDA для обработки массива на GPU. Например, умножьте каждый элемент массива на 2.
// 3. Скопируйте обработанные данные обратно на CPU.
// 4. Замерьте время выполнения обработки на GPU.

#include <iostream>                 // Для ввода и вывода данных
#include <cuda_runtime.h>           // Основная библиотека CUDA (Для CUDA Runtime API)

using namespace std;                // Чтобы не писать std::

// CUDA-ядро (функция, которая выполняется на GPU)
__global__ void multiplyByTwo(float* data, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x; // Глобальный индекс потока
    if (idx < N) {                                   // Проверка выхода за границы
        data[idx] = data[idx] * 2.0f;                // Умножение элемента на 2
    }
}

int main() {                        // Основная функция
    const int N = 1000000;                            // Размер массива
    const int size = N * sizeof(float);               // Размер массива в байтах

    float* h_data = new float[N];                     // Массив в оперативной памяти (CPU)

    // Инициализация массива на CPU
    for (int i = 0; i < N; i++) {
        h_data[i] = i * 1.0f;                         // Заполняем значениями
    }

    float* d_data;                                    // Указатель на память GPU
    cudaMalloc((void**)&d_data, size);                // Выделение памяти на GPU

    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice); // Копирование CPU - GPU

    int threadsPerBlock = 256;                        // Потоков в одном блоке
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock; // Количество блоков

    cudaEvent_t start, stop;                          // CUDA-события для таймера
    cudaEventCreate(&start);                          // Создание события start
    cudaEventCreate(&stop);                           // Создание события stop

    cudaEventRecord(start);                           // Начало замера времени

    multiplyByTwo<<<blocksPerGrid, threadsPerBlock>>>(d_data, N); // Запуск ядра CUDA

    cudaEventRecord(stop);                            // Конец замера
    cudaEventSynchronize(stop);                       // Ожидание завершения GPU

    float milliseconds = 0;                           // Переменная времени
    cudaEventElapsedTime(&milliseconds, start, stop); // Время выполнения ядра

    cudaMemcpy(h_data, d_data, size, cudaMemcpyDeviceToHost); // Копирование GPU - CPU

    cout << "Время выполнения GPU (CUDA): "
         << milliseconds / 1000.0f << " с" << endl;

    // Проверка корректности
    cout << "First 5 elements after processing:" << endl;
    for (int i = 0; i < 5; i++) {
        cout << h_data[i] << " ";
    }
    cout << endl;

    cudaFree(d_data);                                 // Освобождение памяти GPU
    delete[] h_data;                                  // Освобождение памяти CPU

    return 0;                                         // Конец программы
}


Overwriting practice82.cu


In [8]:
# Компиляция
!nvcc practice82.cu -o practice82 -arch=sm_75 -std=c++11            # -arch=sm_75  - архитектура GPU (Tesla T4 в Colab = sm_75)
                                                                    # -std=c++11 — стандарт C++
# Запуск
!./practice82


Время выполнения GPU (CUDA): 9.2128e-05 с
First 5 elements after processing:
0 2 4 6 8 
